# 013 — Figure 5 (axis validation) — PUBLICATION figure, regenerate only

**This is the manuscript Figure 5** (the submitted "v6" composite: Additive axis model, combinatorial states, bridge mechanism, signature specificity, treatment patterns). Load-and-plot from the pre-computed epithelial atlas — **no re-analysis**.

**Revision change:** panel **G** (Signature specificity / AUC) labels enlarged, per the reviewer ("panel G labels difficult to read"). Everything else unchanged from the accepted v6. Output at 1200 dpi.

**Input:** `data/processed_data/pdac_atlas/pdac_epithelial_annotated.h5ad`
**Output:** `.../PANC_atlas_validation/Fig5_axis-validation_revised_1200dpi.png` (+ .pdf)


In [ ]:
import os, numpy as np, scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

DATA_DIR   = "/storage/users/job37yv/Projects/PANC_cancer/data/processed_data/pdac_atlas"
OUTPUT_DIR = "/storage/users/job37yv/Projects/PANC_cancer/code/scripts_beta/PANC_atlas_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

adata_epi = sc.read_h5ad(f"{DATA_DIR}/pdac_epithelial_annotated.h5ad")
print("epithelial:", adata_epi.shape)

# shared colours + helpers (from the analysis notebook)
PURPLE='#8E44AD'; GREEN='#2ECC71'; ORANGE='#F39C12'; RED='#E74C3C'; GREY='#BDC3C7'; BLUE='#3498DB'

def get_expr(ad, gene):
    x = ad[:, gene].X
    return x.toarray().flatten() if hasattr(x, 'toarray') else np.asarray(x).flatten()

def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

def panel_title(ax, letter, subtitle, fontsize_letter=20, fontsize_sub=11):
    ax.set_title(letter, fontsize=fontsize_letter, fontweight='bold', loc='left', pad=8)
    ax.text(0.5, 1.04, subtitle, transform=ax.transAxes, ha='center',
            fontsize=fontsize_sub, fontweight='semibold')


In [ ]:
need=['DiseaseState','Treatment','TreatmentType','phase']
miss=[c for c in need if c not in adata_epi.obs.columns]
print("MISSING obs:",miss) if miss else print("OK — Fig 5 inputs present.")

In [ ]:
###########################################################################
# CELL: VALIDATION FIGURE v8 — overlap fixes, dark labels, tilted bar text
###########################################################################

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import numpy as np

fig = plt.figure(figsize=(20, 28))
gs = GridSpec(4, 3, figure=fig, hspace=0.72, wspace=0.50,
              height_ratios=[0.9, 0.9, 0.9, 0.9])

PURPLE = '#8E44AD'; RED = '#E74C3C'; ORANGE = '#F39C12'
BLUE = '#3498DB'; GREEN = '#2ECC71'; GREY = '#BDC3C7'; DARKGREY = '#7F8C8D'
LABELGREY = '#4D5656'   # NEW: dark grey for annotation text that must stay readable

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.7)
    ax.spines['bottom'].set_linewidth(0.7)
    ax.tick_params(width=0.7)

def to_np(mask):
    return mask.values if hasattr(mask, 'values') else mask

# letter pushed to far top-left, subtitle centered — no more letter/subtitle overlap
def panel_title(ax, letter, subtitle):
    ax.text(-0.12, 1.15, letter, transform=ax.transAxes, fontsize=25,
            fontweight='bold', va='top', ha='left', fontfamily='sans-serif')
    ax.text(0.5, 1.15, subtitle, transform=ax.transAxes, ha='center', va='top',
            fontsize=15, fontweight='semibold')

# Masks
normal_m = adata_epi.obs['DiseaseState'].isin(['Donor','Adjacent normal'])
met_m = adata_epi.obs['DiseaseState']=='Metastatic lesion'
naive_primary = (adata_epi.obs['DiseaseState']=='Primary tumor') & \
                (adata_epi.obs['Treatment']=='Treatment naïve')
gem_m = adata_epi.obs['TreatmentType'].str.contains('Gem|Abraxane', case=False, na=False)
folf_m = adata_epi.obs['TreatmentType'] == 'FOLFIRINOX'
cycling = adata_epi.obs['phase'].isin(['S','G2M'])

# =====================================================================
# ROW 1: METASTASIS
# =====================================================================

# A: Additive axis model
ax = fig.add_subplot(gs[0, 0])
add_names = ['CDK1⁺\nonly', '+CDKN1A', '+WEE1', '+CDKN1A\n+WEE1']
add_met = [29.3, 40.0, 43.9, 58.9]
add_n = [4227, 1534, 1933, 2003]
add_cols = [ORANGE, '#C39BD3', BLUE, PURPLE]

bars = ax.bar(range(4), add_met, color=add_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(add_met, add_n)):
    ax.text(i, v+2.2, f'{v:.1f}%', ha='center', fontsize=15, fontweight='bold')
    # n= now VERTICAL, centred inside the bar so it's clearly readable
    ax.text(i, v/2, f'n={nn:,}', ha='center', va='center', rotation=90,
            fontsize=11, color='white', fontweight='bold')

for i in range(3):
    delta = add_met[i+1] - add_met[i]
    mid_y = (add_met[i] + add_met[i+1]) / 2
    ax.annotate('', xy=(i+1, add_met[i+1]-1), xytext=(i, add_met[i]+1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.3,
                                connectionstyle='arc3,rad=0.2'))
    ax.text(i+0.5, mid_y-1.5, f'+{delta:.0f}%', ha='center', fontsize=11,
            style='italic', color='black')

ax.set_xticks(range(4)); ax.set_xticklabels(add_names, fontsize=13)
ax.set_ylabel('% from metastatic lesions', fontsize=15)
ax.set_ylim(0, 74)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'A', 'Additive axis model\n(among CDK1⁺ cycling cells)')
despine(ax)

# B: All 8 combinatorial states
ax = fig.add_subplot(gs[0, 1])
or_names = ['C⁺A⁺W⁺', 'C⁺A⁻W⁺', 'C⁺A⁺W⁻', 'C⁺A⁻W⁻',
            'C⁻A⁺W⁺', 'C⁻A⁻W⁺', 'C⁻A⁺W⁻', 'C⁻A⁻W⁻']
or_vals = [8.71, 5.33, 4.21, 2.80, 2.57, 1.68, 1.02, 0.39]
met_pcts = [56.5, 43.0, 38.5, 28.2, 24.9, 15.7, 11.9, 6.5]
or_cols = [PURPLE, '#7D3C98', '#AF7AC5', '#D2B4DE',
           BLUE, '#85C1E9', GREY, '#ECF0F1']

y_pos = np.arange(len(or_names))
ax.barh(y_pos, or_vals, color=or_cols, height=0.6, edgecolor='white')
ax.axvline(1, ls=':', c='grey', lw=0.8)

for i, (v, m) in enumerate(zip(or_vals, met_pcts)):
    ax.text(v + 0.20, i, f'OR={v:.2f} ({m:.0f}%)', va='center', fontsize=11.5,
            fontweight='bold' if v > 4 else 'normal')

ax.set_yticks(y_pos)
ax.set_yticklabels(or_names, fontsize=13, family='monospace')
ax.set_xlabel('Odds ratio (met. vs primary naïve)', fontsize=14)
ax.set_xlim(0, max(or_vals) + 3.2)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'B', 'Combinatorial axis states:\nmetastatic enrichment')
ax.invert_yaxis()
despine(ax)

# C: Co-expression gradient
ax = fig.add_subplot(gs[0, 2])
grad_names = ['Normal', 'Primary\nG1', 'Primary\nS', 'Primary\nG2/M', 'Meta-\nstatic']
grad_masks = [
    normal_m,
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G1'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='S'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G2M'),
    met_m,
]
grad_cols = ['#AEB6BF','#85C1E9','#F5B041','#E67E22',RED]
coexpr_pcts = [(adata_epi.obs.loc[m,'coexpr_binary']=='CDK1+/CDKN1A+').mean()*100
               for m in grad_masks]

bars = ax.bar(range(5), coexpr_pcts, color=grad_cols, width=0.65, edgecolor='white')
mx = max(coexpr_pcts)
for i, v in enumerate(coexpr_pcts):
    ax.text(i, v + mx*0.02, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')

for i in range(1, 5):
    if coexpr_pcts[0] > 0:
        fold = coexpr_pcts[i] / coexpr_pcts[0]
        # darker + bigger so the fold labels are actually visible
        ax.text(i, coexpr_pcts[i] + mx*0.09, f'({fold:.0f}×)', ha='center',
                fontsize=11.5, color=LABELGREY, style='italic', fontweight='semibold')

ax.set_ylim(0, mx*1.22)
ax.set_xticks(range(5)); ax.set_xticklabels(grad_names, fontsize=12)
ax.set_ylabel('CDK1⁺/CDKN1A⁺ co-expression (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'C', 'Bottleneck activation along\ndisease progression')
despine(ax)

# =====================================================================
# ROW 2: BRIDGE MECHANISM
# =====================================================================

# D: Correlation flip
ax = fig.add_subplot(gs[1, 0])
states_corr = ['Neither\n(223k)', 'CDKN1A⁺\nonly (53k)',
               'CDK1⁺\nonly (7k)', 'CDK1⁺/CDKN1A⁺\n(4k)']
corr_vals = [-0.062, -0.048, -0.016, 0.114]
corr_cols = [GREY, BLUE, ORANGE, PURPLE]
# separate DARK label colours so the grey "Neither" ρ is readable
corr_txt_cols = ['#34495E', BLUE, '#B9770E', PURPLE]

bars = ax.bar(range(4), corr_vals, color=corr_cols, width=0.55, edgecolor='white')
ax.axhline(0, ls='-', c='black', lw=0.5)

for i, v in enumerate(corr_vals):
    offset = 0.015 if v > 0 else -0.025
    va = 'bottom' if v > 0 else 'top'
    ax.text(i, v+offset, f'ρ = {v:+.3f}', ha='center', va=va, fontsize=12.5,
            fontweight='bold', color=corr_txt_cols[i])

ax.set_xticks(range(4)); ax.set_xticklabels(states_corr, fontsize=11)
ax.set_ylabel('Spearman ρ\n(EMT × Cycling scores)', fontsize=14)
ax.set_ylim(-0.105, 0.19)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'D', 'CDKN1A bridges EMT and\ncell-cycle programs')
despine(ax)

ax.annotate('Programs\ncoupled ↑', xy=(2.75, 0.114), xytext=(1.30, 0.152),
            arrowprops=dict(arrowstyle='->', color=PURPLE, lw=1.5),
            fontsize=11.5, color=PURPLE, fontweight='bold', ha='center')

# E: EMT genes upregulated
ax = fig.add_subplot(gs[1, 1])
emt_genes = ['THBS1','MMP2','SERPINE1','FN1','PLAU','SNAI2','S100A4','VIM']
fc_vals = [1.45, 1.31, 1.28, 1.21, 0.79, 0.55, 0.51, 0.44]

y_pos = np.arange(len(emt_genes))
ax.barh(y_pos, fc_vals, color=PURPLE, height=0.55, alpha=0.85)
ax.axvline(0, ls=':', c='grey', lw=0.8)

for i, v in enumerate(fc_vals):
    ax.text(v+0.04, i, f'+{v:.2f}', va='center', fontsize=12, fontweight='bold',
            color=PURPLE)

ax.set_yticks(y_pos)
ax.set_yticklabels(emt_genes, fontsize=13, style='italic')
ax.set_xlabel('log₂FC (CDK1⁺/CDKN1A⁺ vs CDK1⁺ only)', fontsize=13)
ax.set_xlim(0, max(fc_vals) + 0.30)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'E', 'CDKN1A adds EMT program\nto cycling cells')
ax.invert_yaxis()
despine(ax)

# F: DUAL program
ax = fig.add_subplot(gs[1, 2])
prog_names = ['Neither', 'EMT\nonly', 'Cycling\nonly', 'DUAL\n(EMT+Cyc)']
prog_met_pct = [6.5, 7.8, 8.9, 19.7]
prog_n = [170787, 54656, 54656, 20488]
prog_cols = ['#D5D8DC', RED, ORANGE, PURPLE]

bars = ax.bar(range(4), prog_met_pct, color=prog_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(prog_met_pct, prog_n)):
    ax.text(i, v+0.7, f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
    ax.text(i, -2.6, f'n={nn:,}', ha='center', fontsize=10, color=LABELGREY)

# OR text now VERTICAL and single-line so it fits inside the bar
ax.text(3, 10.0, 'OR=3.54 vs Neither', ha='center', va='center', rotation=90,
        fontsize=12, color='white', fontweight='bold')

ax.set_xticks(range(4)); ax.set_xticklabels(prog_names, fontsize=12)
ax.set_ylabel('% from metastatic lesions', fontsize=14)
ax.set_ylim(-4, 26)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'F', 'DUAL-program cells are\nmost metastatic-prone')
despine(ax)

# =====================================================================
# ROW 3: SPECIFICITY
# =====================================================================

# G: AUC comparison
ax = fig.add_subplot(gs[2, 0])
sig_names = ['Bridge\n(27 genes)', 'Leiden 2\n(19 genes)', 'Generic\nproliferation',
             'Generic\nEMT', 'Bottleneck\nnetwork', 'GEM\nresistance',
             'General\nresistance', 'Axis\n(3 genes)']
sig_aucs = [0.630, 0.620, 0.575, 0.572, 0.555, 0.552, 0.516, 0.515]
sig_cols_f = [PURPLE, PURPLE, GREY, GREY, ORANGE, GREY, GREY, GREY]

y_pos = np.arange(len(sig_names))
ax.barh(y_pos, sig_aucs, color=sig_cols_f, height=0.6, edgecolor='white')
ax.axvline(0.5, ls=':', c='grey', lw=0.8)

for i, v in enumerate(sig_aucs):
    ax.text(v+0.004, i, f'{v:.3f}', va='center', fontsize=13,
            fontweight='bold' if v >= 0.62 else 'normal')

ax.set_yticks(y_pos); ax.set_yticklabels(sig_names, fontsize=13)
ax.set_xlabel('AUC (met. vs naïve, library-corrected)', fontsize=14)
ax.set_xlim(0.47, 0.675)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'G', 'Signature specificity\n(metastatic prediction)')
ax.invert_yaxis()
despine(ax)

# H: Gene expression heatmap
ax = fig.add_subplot(gs[2, 1])
genes_heat = ['CDK1','CDKN1A','WEE1','RRM2','TK1','CLSPN',
              'FN1','TGFBI','S100A4','BIRC5']

gem_cyc_m = gem_m & cycling
gem_g1_m = gem_m & (adata_epi.obs['phase']=='G1')
folf_cyc_m = folf_m & cycling

heat_cols_d = {'Normal': normal_m, 'Naïve': naive_primary,
               'GEM G1': gem_g1_m, 'GEM cyc': gem_cyc_m,
               'FOLF cyc': folf_cyc_m, 'Met': met_m}

heat_matrix = np.zeros((len(genes_heat), len(heat_cols_d)))
for j, (tname, tmask) in enumerate(heat_cols_d.items()):
    for i, gene in enumerate(genes_heat):
        heat_matrix[i,j] = get_expr(adata_epi[tmask], gene).mean()

heat_fc = np.zeros_like(heat_matrix)
for i in range(heat_matrix.shape[0]):
    baseline = heat_matrix[i, 0] + 0.01
    for j in range(heat_matrix.shape[1]):
        heat_fc[i, j] = np.log2((heat_matrix[i, j] + 0.01) / baseline)

im = ax.imshow(heat_fc, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
ax.set_xticks(range(len(heat_cols_d)))
ax.set_xticklabels(heat_cols_d.keys(), fontsize=12.5, rotation=30, ha='right')
ax.set_yticks(range(len(genes_heat)))
ax.set_yticklabels(genes_heat, fontsize=13, style='italic')
cb = plt.colorbar(im, ax=ax, shrink=0.65, aspect=20, pad=0.02)
cb.set_label('log₂FC vs normal', fontsize=14)     # bigger colourbar label
cb.ax.tick_params(labelsize=12)
ax.axhline(5.5, ls='-', c='white', lw=2)
panel_title(ax, 'H', 'Axis gene expression\nacross conditions')

# I: Axis co-detection rates
ax = fig.add_subplot(gs[2, 2])
det_groups = ['Normal', 'Naïve\ncycling', 'GEM\ncycling', 'FOLFIR\ncycling',
              'Met\ncycling']
det_masks = [normal_m,
             naive_primary & cycling,
             gem_cyc_m, folf_cyc_m,
             met_m & cycling]
det_cols = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]

triple_pcts = []; coexpr_pcts_k = []
for mask in det_masks:
    c1 = get_expr(adata_epi[mask], 'CDK1') > 0
    ca = get_expr(adata_epi[mask], 'CDKN1A') > 0
    w = get_expr(adata_epi[mask], 'WEE1') > 0
    triple_pcts.append((c1 & ca & w).mean() * 100)
    coexpr_pcts_k.append((c1 & ca).mean() * 100)

x = np.arange(len(det_groups))
w_bar = 0.35
ax.bar(x - w_bar/2, coexpr_pcts_k, w_bar, color=det_cols, alpha=0.5,
       label='CDK1⁺/CDKN1A⁺')
ax.bar(x + w_bar/2, triple_pcts, w_bar, color=det_cols,
       label='C⁺A⁺W⁺')

# value labels now VERTICAL above each bar → no more twin-label collision
for i in range(len(det_groups)):
    ax.text(i - w_bar/2, coexpr_pcts_k[i]+0.3, f'{coexpr_pcts_k[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')
    ax.text(i + w_bar/2, triple_pcts[i]+0.3, f'{triple_pcts[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')

ax.set_ylim(0, max(coexpr_pcts_k) * 1.35)     # headroom for vertical labels
ax.set_xticks(x); ax.set_xticklabels(det_groups, fontsize=11.5)
ax.set_ylabel('Co-detection rate (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.16),
          ncol=2, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'I', 'Axis co-expression\nacross conditions')
despine(ax)

# =====================================================================
# ROW 4: TREATMENT
# =====================================================================

# J: Drug-specific axis engagement
ax = fig.add_subplot(gs[3, 0])
components = ['CDK1⁺\n(drive)', 'CDKN1A⁺\n(bridge)', 'WEE1⁺\n(brake)', 'DUAL\nprogram']
naive_pct = [8.8, 17.8, 13.8, 13.5]
gem_pct = [3.9, 11.9, 22.8, 3.7]
folf_pct = [16.0, 17.2, 20.7, 28.9]

x = np.arange(len(components))
w = 0.25
ax.bar(x - w, naive_pct, w, color='#85C1E9', label='Naïve cycling')
ax.bar(x, gem_pct, w, color='#9B59B6', label='GEM cycling')
ax.bar(x + w, folf_pct, w, color='#E67E22', label='FOLFIR cycling')

for i, (g, n) in enumerate(zip(gem_pct, naive_pct)):
    if g > n * 1.3:
        ax.text(i, g+1.3, '↑', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')
    elif g < n * 0.7:
        ax.text(i, g+1.3, '↓', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')

for i, (f, n) in enumerate(zip(folf_pct, naive_pct)):
    if f > n * 1.3:
        ax.text(i + w, f+1.3, '↑', ha='center', fontsize=15, color='#E67E22', fontweight='bold')

ax.set_ylim(0, max(folf_pct) * 1.20)
ax.set_xticks(x); ax.set_xticklabels(components, fontsize=12)
ax.set_ylabel('% of cycling cells', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.22),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'J', 'Drug-specific axis engagement\n(GEM: brake ↑ | FOLFIR: drive ↑)')
despine(ax)

# K: Quiescence + WEE1 brake
ax = fig.add_subplot(gs[3, 1])
treat_names = ['Normal', 'Naïve', 'GEM', 'FOLFIR', 'Met.']
cyc_pcts = [45.7, 31.6, 27.0, 37.6, 41.4]
treat_cols_n = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]

bars = ax.bar(range(len(treat_names)), cyc_pcts, color=treat_cols_n,
              width=0.6, edgecolor='white')
for i, v in enumerate(cyc_pcts):
    ax.text(i, v+1.3, f'{v:.1f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold')

wee1_g1_pcts = []
for name, mask in [('Normal', normal_m), ('Naïve', naive_primary),
                     ('GEM', gem_m), ('FOLFIR', folf_m), ('Met', met_m)]:
    g1_mask = mask & (adata_epi.obs['phase']=='G1')
    if g1_mask.sum() > 10:
        w_det = (get_expr(adata_epi[g1_mask], 'WEE1') > 0).mean() * 100
        wee1_g1_pcts.append(w_det)
    else:
        wee1_g1_pcts.append(0)

ax2 = ax.twinx()
ax2.plot(range(len(treat_names)), wee1_g1_pcts, 's--', color='darkblue',
         ms=9, lw=2, label='WEE1⁺ in G1')
# blue labels offset up-LEFT of each marker (offset points) so they clear the bar % labels
for i, v in enumerate(wee1_g1_pcts):
    ax2.annotate(f'{v:.0f}%', (i, v), textcoords='offset points',
                 xytext=(-15, 6), ha='right', fontsize=11.5,
                 color='darkblue', fontweight='semibold')
ax2.set_ylabel('WEE1⁺ in G1 (%)', fontsize=13, color='darkblue')
ax2.tick_params(axis='y', labelsize=11, colors='darkblue')
ax2.spines['top'].set_visible(False)
ax2.set_ylim(top=max(wee1_g1_pcts) * 1.30)     # headroom so top blue label isn't clipped

ax.set_ylim(0, max(cyc_pcts) * 1.20)
ax.set_xticks(range(len(treat_names)))
ax.set_xticklabels(treat_names, fontsize=12)
ax.set_ylabel('% cycling (S+G2M)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax2.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
           framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'K', 'GEM: quiescence + WEE1 brake\n(73% G1, WEE1⁺ in quiescent)')
despine(ax)

# L: Bottleneck enrichment
ax = fig.add_subplot(gs[3, 2])
pcts_plot = [50, 60, 70, 75, 80, 85, 90, 95]
enr_data = {
    'GEM cycling': ([50.2,39.2,30.8,28.2,24.4,20.1,13.8,5.9], '#9B59B6'),
    'FOLFIR cycling': ([54.9,45.8,39.2,35.6,30.6,25.1,18.0,9.7], '#E67E22'),
    'Metastatic': ([55.5,48.2,39.3,31.6,24.9,18.8,13.6,8.7], RED),
}
expected = [50, 40, 30, 25, 20, 15, 10, 5]

for name, (vals, col) in enr_data.items():
    fold = [v/e for v, e in zip(vals, expected)]
    ax.plot(pcts_plot, fold, 'o-', color=col, lw=2.5, ms=7, label=name)

ax.axhline(1, ls=':', c='grey', lw=0.8)

ax.set_xlabel('Axis score percentile threshold', fontsize=14)
ax.set_ylabel('Fold enrichment vs naïve', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.set_ylim(0.7, 2.5)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'L', 'Bottleneck enrichment\nin survivors & metastasis')
despine(ax)

ax.annotate('FOLFIR P90:\nOR=1.97', xy=(90, 1.80), xytext=(69, 2.28),
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=1.5),
            fontsize=12, color='#E67E22', fontweight='bold')
ax.annotate('Met P90:\nOR=1.42', xy=(90, 1.36), xytext=(69, 1.60),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5),
            fontsize=12, color=RED, fontweight='bold')

# =====================================================================
# SUPTITLE + ROW LABELS
# =====================================================================

fig.suptitle(
    'External validation: CDK1–CDKN1A–WEE1 bottleneck axis\n'
    '300,577 epithelial cells · 231 patients · 12 independent studies',
    fontsize=19, fontweight='bold', y=1.005, fontfamily='sans-serif')

row_labels = [
    'Metastatic\nassociation',
    'Bridge\nmechanism',
    'Signature\nspecificity',
    'Treatment\npatterns',
]
for i, label in enumerate(row_labels):
    fig.text(1.005, 0.86 - i*0.235, label, fontsize=13, fontweight='bold',
             rotation=270, va='center', ha='center', color=DARKGREY,
             style='italic', fontfamily='sans-serif')

# =====================================================================
# SAVE 1200 DPI
# =====================================================================
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised_1200dpi.png", dpi=1200,
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised.pdf",
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Saved 1200 DPI validation figure to {OUTPUT_DIR}/")

In [ ]:
###########################################################################
# CELL: VALIDATION FIGURE v10 — final A/D/K label nudges
###########################################################################

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import numpy as np

fig = plt.figure(figsize=(20, 28))
gs = GridSpec(4, 3, figure=fig, hspace=0.72, wspace=0.50,
              height_ratios=[0.9, 0.9, 0.9, 0.9])

PURPLE = '#8E44AD'; RED = '#E74C3C'; ORANGE = '#F39C12'
BLUE = '#3498DB'; GREEN = '#2ECC71'; GREY = '#BDC3C7'; DARKGREY = '#7F8C8D'
LABELGREY = '#4D5656'   # dark grey for annotation text that must stay readable

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.7)
    ax.spines['bottom'].set_linewidth(0.7)
    ax.tick_params(width=0.7)

def to_np(mask):
    return mask.values if hasattr(mask, 'values') else mask

# letter pushed to far top-left, subtitle centered — no more letter/subtitle overlap
def panel_title(ax, letter, subtitle):
    ax.text(-0.12, 1.15, letter, transform=ax.transAxes, fontsize=25,
            fontweight='bold', va='top', ha='left', fontfamily='sans-serif')
    ax.text(0.5, 1.15, subtitle, transform=ax.transAxes, ha='center', va='top',
            fontsize=15, fontweight='semibold')

# Masks
normal_m = adata_epi.obs['DiseaseState'].isin(['Donor','Adjacent normal'])
met_m = adata_epi.obs['DiseaseState']=='Metastatic lesion'
naive_primary = (adata_epi.obs['DiseaseState']=='Primary tumor') & \
                (adata_epi.obs['Treatment']=='Treatment naïve')
gem_m = adata_epi.obs['TreatmentType'].str.contains('Gem|Abraxane', case=False, na=False)
folf_m = adata_epi.obs['TreatmentType'] == 'FOLFIRINOX'
cycling = adata_epi.obs['phase'].isin(['S','G2M'])

# =====================================================================
# ROW 1: METASTASIS
# =====================================================================

# A: Additive axis model
ax = fig.add_subplot(gs[0, 0])
add_names = ['CDK1⁺\nonly', '+CDKN1A', '+WEE1', '+CDKN1A\n+WEE1']
add_met = [29.3, 40.0, 43.9, 58.9]
add_n = [4227, 1534, 1933, 2003]
add_cols = [ORANGE, '#C39BD3', BLUE, PURPLE]

bars = ax.bar(range(4), add_met, color=add_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(add_met, add_n)):
    # value % lifted higher above each bar
    ax.text(i, v+4.0, f'{v:.1f}%', ha='center', fontsize=15, fontweight='bold')
    # n= vertical, centred inside the bar
    ax.text(i, v/2, f'n={nn:,}', ha='center', va='center', rotation=90,
            fontsize=11, color='white', fontweight='bold')

for i in range(3):
    delta = add_met[i+1] - add_met[i]
    ax.annotate('', xy=(i+1, add_met[i+1]-1), xytext=(i, add_met[i]+1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.3,
                                connectionstyle='arc3,rad=0.2'))
    # delta text in white space above the higher of the two bars
    ax.text(i+0.5, max(add_met[i], add_met[i+1]) + 3.5, f'+{delta:.0f}%',
            ha='center', fontsize=11, style='italic', color='black')

ax.set_xticks(range(4)); ax.set_xticklabels(add_names, fontsize=13)
ax.set_ylabel('% from metastatic lesions', fontsize=15)
ax.set_ylim(0, 78)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'A', 'Additive axis model\n(among CDK1⁺ cycling cells)')
despine(ax)

# B: All 8 combinatorial states
ax = fig.add_subplot(gs[0, 1])
or_names = ['C⁺A⁺W⁺', 'C⁺A⁻W⁺', 'C⁺A⁺W⁻', 'C⁺A⁻W⁻',
            'C⁻A⁺W⁺', 'C⁻A⁻W⁺', 'C⁻A⁺W⁻', 'C⁻A⁻W⁻']
or_vals = [8.71, 5.33, 4.21, 2.80, 2.57, 1.68, 1.02, 0.39]
met_pcts = [56.5, 43.0, 38.5, 28.2, 24.9, 15.7, 11.9, 6.5]
or_cols = [PURPLE, '#7D3C98', '#AF7AC5', '#D2B4DE',
           BLUE, '#85C1E9', GREY, '#ECF0F1']

y_pos = np.arange(len(or_names))
ax.barh(y_pos, or_vals, color=or_cols, height=0.6, edgecolor='white')
ax.axvline(1, ls=':', c='grey', lw=0.8)

for i, (v, m) in enumerate(zip(or_vals, met_pcts)):
    ax.text(v + 0.20, i, f'OR={v:.2f} ({m:.0f}%)', va='center', fontsize=11.5,
            fontweight='bold' if v > 4 else 'normal')

ax.set_yticks(y_pos)
ax.set_yticklabels(or_names, fontsize=13, family='monospace')
ax.set_xlabel('Odds ratio (met. vs primary naïve)', fontsize=14)
ax.set_xlim(0, max(or_vals) + 3.2)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'B', 'Combinatorial axis states:\nmetastatic enrichment')
ax.invert_yaxis()
despine(ax)

# C: Co-expression gradient
ax = fig.add_subplot(gs[0, 2])
grad_names = ['Normal', 'Primary\nG1', 'Primary\nS', 'Primary\nG2/M', 'Meta-\nstatic']
grad_masks = [
    normal_m,
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G1'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='S'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G2M'),
    met_m,
]
grad_cols = ['#AEB6BF','#85C1E9','#F5B041','#E67E22',RED]
coexpr_pcts = [(adata_epi.obs.loc[m,'coexpr_binary']=='CDK1+/CDKN1A+').mean()*100
               for m in grad_masks]

bars = ax.bar(range(5), coexpr_pcts, color=grad_cols, width=0.65, edgecolor='white')
mx = max(coexpr_pcts)
for i, v in enumerate(coexpr_pcts):
    ax.text(i, v + mx*0.02, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')

for i in range(1, 5):
    if coexpr_pcts[0] > 0:
        fold = coexpr_pcts[i] / coexpr_pcts[0]
        ax.text(i, coexpr_pcts[i] + mx*0.09, f'({fold:.0f}×)', ha='center',
                fontsize=11.5, color=LABELGREY, style='italic', fontweight='semibold')

ax.set_ylim(0, mx*1.22)
ax.set_xticks(range(5)); ax.set_xticklabels(grad_names, fontsize=12)
ax.set_ylabel('CDK1⁺/CDKN1A⁺ co-expression (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'C', 'Bottleneck activation along\ndisease progression')
despine(ax)

# =====================================================================
# ROW 2: BRIDGE MECHANISM
# =====================================================================

# D: Correlation flip
ax = fig.add_subplot(gs[1, 0])
states_corr = ['Neither\n(223k)', 'CDKN1A⁺\nonly (53k)',
               'CDK1⁺\nonly (7k)', 'CDK1⁺/CDKN1A⁺\n(4k)']
corr_vals = [-0.062, -0.048, -0.016, 0.114]
corr_cols = [GREY, BLUE, ORANGE, PURPLE]
# separate DARK label colours so the grey "Neither" ρ is readable
corr_txt_cols = ['#34495E', BLUE, '#B9770E', PURPLE]

bars = ax.bar(range(4), corr_vals, color=corr_cols, width=0.55, edgecolor='white')
ax.axhline(0, ls='-', c='black', lw=0.5)

for i, v in enumerate(corr_vals):
    # labels sit close to the bar end (small offset), number on 2nd line
    offset = 0.008 if v > 0 else -0.012
    va = 'bottom' if v > 0 else 'top'
    ax.text(i, v+offset, f'ρ =\n{v:+.3f}', ha='center', va=va, fontsize=12.5,
            fontweight='bold', color=corr_txt_cols[i], linespacing=0.95)

ax.set_xticks(range(4)); ax.set_xticklabels(states_corr, fontsize=11)
ax.set_ylabel('Spearman ρ\n(EMT × Cycling scores)', fontsize=14)
ax.set_ylim(-0.115, 0.19)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'D', 'CDKN1A bridges EMT and\ncell-cycle programs')
despine(ax)

ax.annotate('Programs\ncoupled ↑', xy=(2.75, 0.114), xytext=(1.30, 0.152),
            arrowprops=dict(arrowstyle='->', color=PURPLE, lw=1.5),
            fontsize=11.5, color=PURPLE, fontweight='bold', ha='center')

# E: EMT genes upregulated
ax = fig.add_subplot(gs[1, 1])
emt_genes = ['THBS1','MMP2','SERPINE1','FN1','PLAU','SNAI2','S100A4','VIM']
fc_vals = [1.45, 1.31, 1.28, 1.21, 0.79, 0.55, 0.51, 0.44]

y_pos = np.arange(len(emt_genes))
ax.barh(y_pos, fc_vals, color=PURPLE, height=0.55, alpha=0.85)
ax.axvline(0, ls=':', c='grey', lw=0.8)

for i, v in enumerate(fc_vals):
    ax.text(v+0.04, i, f'+{v:.2f}', va='center', fontsize=12, fontweight='bold',
            color=PURPLE)

ax.set_yticks(y_pos)
ax.set_yticklabels(emt_genes, fontsize=13, style='italic')
ax.set_xlabel('log₂FC (CDK1⁺/CDKN1A⁺ vs CDK1⁺ only)', fontsize=13)
ax.set_xlim(0, max(fc_vals) + 0.30)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'E', 'CDKN1A adds EMT program\nto cycling cells')
ax.invert_yaxis()
despine(ax)

# F: DUAL program
ax = fig.add_subplot(gs[1, 2])
prog_names = ['Neither', 'EMT\nonly', 'Cycling\nonly', 'DUAL\n(EMT+Cyc)']
prog_met_pct = [6.5, 7.8, 8.9, 19.7]
prog_n = [170787, 54656, 54656, 20488]
prog_cols = ['#D5D8DC', RED, ORANGE, PURPLE]

bars = ax.bar(range(4), prog_met_pct, color=prog_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(prog_met_pct, prog_n)):
    ax.text(i, v+0.7, f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
    ax.text(i, -2.6, f'n={nn:,}', ha='center', fontsize=10, color=LABELGREY)

# OR text vertical and single-line so it fits inside the bar
ax.text(3, 10.0, 'OR=3.54 vs Neither', ha='center', va='center', rotation=90,
        fontsize=12, color='white', fontweight='bold')

ax.set_xticks(range(4)); ax.set_xticklabels(prog_names, fontsize=12)
ax.set_ylabel('% from metastatic lesions', fontsize=14)
ax.set_ylim(-4, 26)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'F', 'DUAL-program cells are\nmost metastatic-prone')
despine(ax)

# =====================================================================
# ROW 3: SPECIFICITY
# =====================================================================

# G: AUC comparison
ax = fig.add_subplot(gs[2, 0])
sig_names = ['Bridge\n(27 genes)', 'Leiden 2\n(19 genes)', 'Generic\nproliferation',
             'Generic\nEMT', 'Bottleneck\nnetwork', 'GEM\nresistance',
             'General\nresistance', 'Axis\n(3 genes)']
sig_aucs = [0.630, 0.620, 0.575, 0.572, 0.555, 0.552, 0.516, 0.515]
sig_cols_f = [PURPLE, PURPLE, GREY, GREY, ORANGE, GREY, GREY, GREY]

y_pos = np.arange(len(sig_names))
ax.barh(y_pos, sig_aucs, color=sig_cols_f, height=0.6, edgecolor='white')
ax.axvline(0.5, ls=':', c='grey', lw=0.8)

for i, v in enumerate(sig_aucs):
    ax.text(v+0.004, i, f'{v:.3f}', va='center', fontsize=13,
            fontweight='bold' if v >= 0.62 else 'normal')

ax.set_yticks(y_pos); ax.set_yticklabels(sig_names, fontsize=13)
ax.set_xlabel('AUC (met. vs naïve, library-corrected)', fontsize=14)
ax.set_xlim(0.47, 0.675)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'G', 'Signature specificity\n(metastatic prediction)')
ax.invert_yaxis()
despine(ax)

# H: Gene expression heatmap
ax = fig.add_subplot(gs[2, 1])
genes_heat = ['CDK1','CDKN1A','WEE1','RRM2','TK1','CLSPN',
              'FN1','TGFBI','S100A4','BIRC5']

gem_cyc_m = gem_m & cycling
gem_g1_m = gem_m & (adata_epi.obs['phase']=='G1')
folf_cyc_m = folf_m & cycling

heat_cols_d = {'Normal': normal_m, 'Naïve': naive_primary,
               'GEM G1': gem_g1_m, 'GEM cyc': gem_cyc_m,
               'FOLF cyc': folf_cyc_m, 'Met': met_m}

heat_matrix = np.zeros((len(genes_heat), len(heat_cols_d)))
for j, (tname, tmask) in enumerate(heat_cols_d.items()):
    for i, gene in enumerate(genes_heat):
        heat_matrix[i,j] = get_expr(adata_epi[tmask], gene).mean()

heat_fc = np.zeros_like(heat_matrix)
for i in range(heat_matrix.shape[0]):
    baseline = heat_matrix[i, 0] + 0.01
    for j in range(heat_matrix.shape[1]):
        heat_fc[i, j] = np.log2((heat_matrix[i, j] + 0.01) / baseline)

im = ax.imshow(heat_fc, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
ax.set_xticks(range(len(heat_cols_d)))
ax.set_xticklabels(heat_cols_d.keys(), fontsize=12.5, rotation=30, ha='right')
ax.set_yticks(range(len(genes_heat)))
ax.set_yticklabels(genes_heat, fontsize=13, style='italic')
cb = plt.colorbar(im, ax=ax, shrink=0.65, aspect=20, pad=0.02)
cb.set_label('log₂FC vs normal', fontsize=14)
cb.ax.tick_params(labelsize=12)
ax.axhline(5.5, ls='-', c='white', lw=2)
panel_title(ax, 'H', 'Axis gene expression\nacross conditions')

# I: Axis co-detection rates
ax = fig.add_subplot(gs[2, 2])
det_groups = ['Normal', 'Naïve\ncycling', 'GEM\ncycling', 'FOLFIR\ncycling',
              'Met\ncycling']
det_masks = [normal_m,
             naive_primary & cycling,
             gem_cyc_m, folf_cyc_m,
             met_m & cycling]
det_cols = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]

triple_pcts = []; coexpr_pcts_k = []
for mask in det_masks:
    c1 = get_expr(adata_epi[mask], 'CDK1') > 0
    ca = get_expr(adata_epi[mask], 'CDKN1A') > 0
    w = get_expr(adata_epi[mask], 'WEE1') > 0
    triple_pcts.append((c1 & ca & w).mean() * 100)
    coexpr_pcts_k.append((c1 & ca).mean() * 100)

x = np.arange(len(det_groups))
w_bar = 0.35
ax.bar(x - w_bar/2, coexpr_pcts_k, w_bar, color=det_cols, alpha=0.5,
       label='CDK1⁺/CDKN1A⁺')
ax.bar(x + w_bar/2, triple_pcts, w_bar, color=det_cols,
       label='C⁺A⁺W⁺')

# value labels vertical above each bar → no more twin-label collision
for i in range(len(det_groups)):
    ax.text(i - w_bar/2, coexpr_pcts_k[i]+0.3, f'{coexpr_pcts_k[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')
    ax.text(i + w_bar/2, triple_pcts[i]+0.3, f'{triple_pcts[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')

ax.set_ylim(0, max(coexpr_pcts_k) * 1.35)     # headroom for vertical labels
ax.set_xticks(x); ax.set_xticklabels(det_groups, fontsize=11.5)
ax.set_ylabel('Co-detection rate (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.16),
          ncol=2, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'I', 'Axis co-expression\nacross conditions')
despine(ax)

# =====================================================================
# ROW 4: TREATMENT
# =====================================================================

# J: Drug-specific axis engagement
ax = fig.add_subplot(gs[3, 0])
components = ['CDK1⁺\n(drive)', 'CDKN1A⁺\n(bridge)', 'WEE1⁺\n(brake)', 'DUAL\nprogram']
naive_pct = [8.8, 17.8, 13.8, 13.5]
gem_pct = [3.9, 11.9, 22.8, 3.7]
folf_pct = [16.0, 17.2, 20.7, 28.9]

x = np.arange(len(components))
w = 0.25
ax.bar(x - w, naive_pct, w, color='#85C1E9', label='Naïve cycling')
ax.bar(x, gem_pct, w, color='#9B59B6', label='GEM cycling')
ax.bar(x + w, folf_pct, w, color='#E67E22', label='FOLFIR cycling')

for i, (g, n) in enumerate(zip(gem_pct, naive_pct)):
    if g > n * 1.3:
        ax.text(i, g+1.3, '↑', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')
    elif g < n * 0.7:
        ax.text(i, g+1.3, '↓', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')

for i, (f, n) in enumerate(zip(folf_pct, naive_pct)):
    if f > n * 1.3:
        ax.text(i + w, f+1.3, '↑', ha='center', fontsize=15, color='#E67E22', fontweight='bold')

ax.set_ylim(0, max(folf_pct) * 1.20)
ax.set_xticks(x); ax.set_xticklabels(components, fontsize=12)
ax.set_ylabel('% of cycling cells', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.22),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'J', 'Drug-specific axis engagement\n(GEM: brake ↑ | FOLFIR: drive ↑)')
despine(ax)

# K: Quiescence + WEE1 brake
ax = fig.add_subplot(gs[3, 1])
treat_names = ['Normal', 'Naïve', 'GEM', 'FOLFIR', 'Met.']
cyc_pcts = [45.7, 31.6, 27.0, 37.6, 41.4]
treat_cols_n = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]

bars = ax.bar(range(len(treat_names)), cyc_pcts, color=treat_cols_n,
              width=0.6, edgecolor='white')
for i, v in enumerate(cyc_pcts):
    ax.text(i, v+1.3, f'{v:.1f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold')

wee1_g1_pcts = []
for name, mask in [('Normal', normal_m), ('Naïve', naive_primary),
                     ('GEM', gem_m), ('FOLFIR', folf_m), ('Met', met_m)]:
    g1_mask = mask & (adata_epi.obs['phase']=='G1')
    if g1_mask.sum() > 10:
        w_det = (get_expr(adata_epi[g1_mask], 'WEE1') > 0).mean() * 100
        wee1_g1_pcts.append(w_det)
    else:
        wee1_g1_pcts.append(0)

ax2 = ax.twinx()
ax2.plot(range(len(treat_names)), wee1_g1_pcts, 's--', color='darkblue',
         ms=9, lw=2, label='WEE1⁺ in G1')
ax2.set_ylim(top=max(wee1_g1_pcts) * 1.32)     # headroom first
# per-marker manual placement (offset points): (dx, dy, ha, va)
#   Normal, Naïve → above; GEM → above; FOLFIR(18%) → below; Met → further below
blue_offsets = [
    (0,  9, 'center', 'bottom'),   # Normal  9%  above
    (0,  9, 'center', 'bottom'),   # Naïve  15%  above
    (0, 11, 'center', 'bottom'),   # GEM    31%  above the square
    (0,-13, 'center', 'top'),      # FOLFIR 18%  below the square
    (0,-20, 'center', 'top'),      # Met    31%  further below the square
]
for i, v in enumerate(wee1_g1_pcts):
    dx, dy, ha, va = blue_offsets[i]
    ax2.annotate(f'{v:.0f}%', (i, v), textcoords='offset points',
                 xytext=(dx, dy), ha=ha, va=va, fontsize=11.5,
                 color='darkblue', fontweight='semibold')
ax2.set_ylabel('WEE1⁺ in G1 (%)', fontsize=13, color='darkblue')
ax2.tick_params(axis='y', labelsize=11, colors='darkblue')
ax2.spines['top'].set_visible(False)

ax.set_ylim(0, max(cyc_pcts) * 1.20)
ax.set_xticks(range(len(treat_names)))
ax.set_xticklabels(treat_names, fontsize=12)
ax.set_ylabel('% cycling (S+G2M)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax2.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
           framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'K', 'GEM: quiescence + WEE1 brake\n(73% G1, WEE1⁺ in quiescent)')
despine(ax)

# L: Bottleneck enrichment
ax = fig.add_subplot(gs[3, 2])
pcts_plot = [50, 60, 70, 75, 80, 85, 90, 95]
enr_data = {
    'GEM cycling': ([50.2,39.2,30.8,28.2,24.4,20.1,13.8,5.9], '#9B59B6'),
    'FOLFIR cycling': ([54.9,45.8,39.2,35.6,30.6,25.1,18.0,9.7], '#E67E22'),
    'Metastatic': ([55.5,48.2,39.3,31.6,24.9,18.8,13.6,8.7], RED),
}
expected = [50, 40, 30, 25, 20, 15, 10, 5]

for name, (vals, col) in enr_data.items():
    fold = [v/e for v, e in zip(vals, expected)]
    ax.plot(pcts_plot, fold, 'o-', color=col, lw=2.5, ms=7, label=name)

ax.axhline(1, ls=':', c='grey', lw=0.8)

ax.set_xlabel('Axis score percentile threshold', fontsize=14)
ax.set_ylabel('Fold enrichment vs naïve', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.set_ylim(0.7, 2.5)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'L', 'Bottleneck enrichment\nin survivors & metastasis')
despine(ax)

ax.annotate('FOLFIR P90:\nOR=1.97', xy=(90, 1.80), xytext=(69, 2.28),
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=1.5),
            fontsize=12, color='#E67E22', fontweight='bold')
ax.annotate('Met P90:\nOR=1.42', xy=(90, 1.36), xytext=(69, 1.60),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5),
            fontsize=12, color=RED, fontweight='bold')

# =====================================================================
# SUPTITLE + ROW LABELS
# =====================================================================

fig.suptitle(
    'External validation: CDK1–CDKN1A–WEE1 bottleneck axis\n'
    '300,577 epithelial cells · 231 patients · 12 independent studies',
    fontsize=19, fontweight='bold', y=1.005, fontfamily='sans-serif')

row_labels = [
    'Metastatic\nassociation',
    'Bridge\nmechanism',
    'Signature\nspecificity',
    'Treatment\npatterns',
]
for i, label in enumerate(row_labels):
    fig.text(1.005, 0.86 - i*0.235, label, fontsize=13, fontweight='bold',
             rotation=270, va='center', ha='center', color=DARKGREY,
             style='italic', fontfamily='sans-serif')

# =====================================================================
# SAVE 1200 DPI
# =====================================================================
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised_1200dpi.png", dpi=1200,
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised.pdf",
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Saved 1200 DPI validation figure to {OUTPUT_DIR}/")

In [ ]:
###########################################################################
# CELL: VALIDATION FIGURE v10 — final A/D/K label nudges
###########################################################################
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import numpy as np
fig = plt.figure(figsize=(20, 28))
gs = GridSpec(4, 3, figure=fig, hspace=0.72, wspace=0.50,
              height_ratios=[0.9, 0.9, 0.9, 0.9])
PURPLE = '#8E44AD'; RED = '#E74C3C'; ORANGE = '#F39C12'
BLUE = '#3498DB'; GREEN = '#2ECC71'; GREY = '#BDC3C7'; DARKGREY = '#7F8C8D'
LABELGREY = '#4D5656'   # dark grey for annotation text that must stay readable
def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.7)
    ax.spines['bottom'].set_linewidth(0.7)
    ax.tick_params(width=0.7)
def to_np(mask):
    return mask.values if hasattr(mask, 'values') else mask
# letter pushed to far top-left, subtitle centered — no more letter/subtitle overlap
def panel_title(ax, letter, subtitle):
    ax.text(-0.12, 1.15, letter, transform=ax.transAxes, fontsize=25,
            fontweight='bold', va='top', ha='left', fontfamily='sans-serif')
    ax.text(0.5, 1.15, subtitle, transform=ax.transAxes, ha='center', va='top',
            fontsize=15, fontweight='semibold')
# Masks
normal_m = adata_epi.obs['DiseaseState'].isin(['Donor','Adjacent normal'])
met_m = adata_epi.obs['DiseaseState']=='Metastatic lesion'
naive_primary = (adata_epi.obs['DiseaseState']=='Primary tumor') & \
                (adata_epi.obs['Treatment']=='Treatment naïve')
gem_m = adata_epi.obs['TreatmentType'].str.contains('Gem|Abraxane', case=False, na=False)
folf_m = adata_epi.obs['TreatmentType'] == 'FOLFIRINOX'
cycling = adata_epi.obs['phase'].isin(['S','G2M'])
# =====================================================================
# ROW 1: METASTASIS
# =====================================================================
# A: Additive axis model
ax = fig.add_subplot(gs[0, 0])
add_names = ['CDK1⁺\nonly', '+CDKN1A', '+WEE1', '+CDKN1A\n+WEE1']
add_met = [29.3, 40.0, 43.9, 58.9]
add_n = [4227, 1534, 1933, 2003]
add_cols = [ORANGE, '#C39BD3', BLUE, PURPLE]
bars = ax.bar(range(4), add_met, color=add_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(add_met, add_n)):
    # value % lifted higher above each bar
    ax.text(i, v+4.0, f'{v:.1f}%', ha='center', fontsize=15, fontweight='bold')
    # n= vertical, centred inside the bar
    ax.text(i, v/2, f'n={nn:,}', ha='center', va='center', rotation=90,
            fontsize=11, color='white', fontweight='bold')
for i in range(3):
    delta = add_met[i+1] - add_met[i]
    ax.annotate('', xy=(i+1, add_met[i+1]-1), xytext=(i, add_met[i]+1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.3,
                                connectionstyle='arc3,rad=0.2'))
    # delta % nudged UP + LEFT (was i+0.5 / +3.5) to clear the arrow and value labels
    ax.text(i+0.25, max(add_met[i], add_met[i+1]) + 7.0, f'+{delta:.0f}%',
            ha='center', fontsize=11, style='italic', color='black')
ax.set_xticks(range(4)); ax.set_xticklabels(add_names, fontsize=13)
ax.set_ylabel('% from metastatic lesions', fontsize=15)
ax.set_ylim(0, 78)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'A', 'Additive axis model\n(among CDK1⁺ cycling cells)')
despine(ax)
# B: All 8 combinatorial states
ax = fig.add_subplot(gs[0, 1])
or_names = ['C⁺A⁺W⁺', 'C⁺A⁻W⁺', 'C⁺A⁺W⁻', 'C⁺A⁻W⁻',
            'C⁻A⁺W⁺', 'C⁻A⁻W⁺', 'C⁻A⁺W⁻', 'C⁻A⁻W⁻']
or_vals = [8.71, 5.33, 4.21, 2.80, 2.57, 1.68, 1.02, 0.39]
met_pcts = [56.5, 43.0, 38.5, 28.2, 24.9, 15.7, 11.9, 6.5]
or_cols = [PURPLE, '#7D3C98', '#AF7AC5', '#D2B4DE',
           BLUE, '#85C1E9', GREY, '#ECF0F1']
y_pos = np.arange(len(or_names))
ax.barh(y_pos, or_vals, color=or_cols, height=0.6, edgecolor='white')
ax.axvline(1, ls=':', c='grey', lw=0.8)
for i, (v, m) in enumerate(zip(or_vals, met_pcts)):
    ax.text(v + 0.20, i, f'OR={v:.2f} ({m:.0f}%)', va='center', fontsize=11.5,
            fontweight='bold' if v > 4 else 'normal')
ax.set_yticks(y_pos)
ax.set_yticklabels(or_names, fontsize=13, family='monospace')
ax.set_xlabel('Odds ratio (met. vs primary naïve)', fontsize=14)
ax.set_xlim(0, max(or_vals) + 3.2)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'B', 'Combinatorial axis states:\nmetastatic enrichment')
ax.invert_yaxis()
despine(ax)
# C: Co-expression gradient
ax = fig.add_subplot(gs[0, 2])
grad_names = ['Normal', 'Primary\nG1', 'Primary\nS', 'Primary\nG2/M', 'Meta-\nstatic']
grad_masks = [
    normal_m,
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G1'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='S'),
    (adata_epi.obs['DiseaseState']=='Primary tumor') & (adata_epi.obs['phase']=='G2M'),
    met_m,
]
grad_cols = ['#AEB6BF','#85C1E9','#F5B041','#E67E22',RED]
coexpr_pcts = [(adata_epi.obs.loc[m,'coexpr_binary']=='CDK1+/CDKN1A+').mean()*100
               for m in grad_masks]
bars = ax.bar(range(5), coexpr_pcts, color=grad_cols, width=0.65, edgecolor='white')
mx = max(coexpr_pcts)
for i, v in enumerate(coexpr_pcts):
    ax.text(i, v + mx*0.02, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
for i in range(1, 5):
    if coexpr_pcts[0] > 0:
        fold = coexpr_pcts[i] / coexpr_pcts[0]
        ax.text(i, coexpr_pcts[i] + mx*0.09, f'({fold:.0f}×)', ha='center',
                fontsize=11.5, color=LABELGREY, style='italic', fontweight='semibold')
ax.set_ylim(0, mx*1.22)
ax.set_xticks(range(5)); ax.set_xticklabels(grad_names, fontsize=12)
ax.set_ylabel('CDK1⁺/CDKN1A⁺ co-expression (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'C', 'Bottleneck activation along\ndisease progression')
despine(ax)
# =====================================================================
# ROW 2: BRIDGE MECHANISM
# =====================================================================
# D: Correlation flip
ax = fig.add_subplot(gs[1, 0])
states_corr = ['Neither\n(223k)', 'CDKN1A⁺\nonly (53k)',
               'CDK1⁺\nonly (7k)', 'CDK1⁺/CDKN1A⁺\n(4k)']
corr_vals = [-0.062, -0.048, -0.016, 0.114]
corr_cols = [GREY, BLUE, ORANGE, PURPLE]
# separate DARK label colours so the grey "Neither" ρ is readable
corr_txt_cols = ['#34495E', BLUE, '#B9770E', PURPLE]
bars = ax.bar(range(4), corr_vals, color=corr_cols, width=0.55, edgecolor='white')
ax.axhline(0, ls='-', c='black', lw=0.5)
for i, v in enumerate(corr_vals):
    # labels sit close to the bar end (small offset), number on 2nd line
    offset = 0.008 if v > 0 else -0.012
    va = 'bottom' if v > 0 else 'top'
    ax.text(i, v+offset, f'ρ =\n{v:+.3f}', ha='center', va=va, fontsize=12.5,
            fontweight='bold', color=corr_txt_cols[i], linespacing=0.95)
ax.set_xticks(range(4)); ax.set_xticklabels(states_corr, fontsize=11)
ax.set_ylabel('Spearman ρ\n(EMT × Cycling scores)', fontsize=14)
ax.set_ylim(-0.115, 0.19)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'D', 'CDKN1A bridges EMT and\ncell-cycle programs')
despine(ax)
ax.annotate('Programs\ncoupled ↑', xy=(2.75, 0.114), xytext=(1.30, 0.152),
            arrowprops=dict(arrowstyle='->', color=PURPLE, lw=1.5),
            fontsize=11.5, color=PURPLE, fontweight='bold', ha='center')
# E: EMT genes upregulated
ax = fig.add_subplot(gs[1, 1])
emt_genes = ['THBS1','MMP2','SERPINE1','FN1','PLAU','SNAI2','S100A4','VIM']
fc_vals = [1.45, 1.31, 1.28, 1.21, 0.79, 0.55, 0.51, 0.44]
y_pos = np.arange(len(emt_genes))
ax.barh(y_pos, fc_vals, color=PURPLE, height=0.55, alpha=0.85)
ax.axvline(0, ls=':', c='grey', lw=0.8)
for i, v in enumerate(fc_vals):
    ax.text(v+0.04, i, f'+{v:.2f}', va='center', fontsize=12, fontweight='bold',
            color=PURPLE)
ax.set_yticks(y_pos)
ax.set_yticklabels(emt_genes, fontsize=13, style='italic')
ax.set_xlabel('log₂FC (CDK1⁺/CDKN1A⁺ vs CDK1⁺ only)', fontsize=13)
ax.set_xlim(0, max(fc_vals) + 0.30)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'E', 'CDKN1A adds EMT program\nto cycling cells')
ax.invert_yaxis()
despine(ax)
# F: DUAL program
ax = fig.add_subplot(gs[1, 2])
prog_names = ['Neither', 'EMT\nonly', 'Cycling\nonly', 'DUAL\n(EMT+Cyc)']
prog_met_pct = [6.5, 7.8, 8.9, 19.7]
prog_n = [170787, 54656, 54656, 20488]
prog_cols = ['#D5D8DC', RED, ORANGE, PURPLE]
bars = ax.bar(range(4), prog_met_pct, color=prog_cols, width=0.6, edgecolor='white')
for i, (v, nn) in enumerate(zip(prog_met_pct, prog_n)):
    ax.text(i, v+0.7, f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
    ax.text(i, -2.6, f'n={nn:,}', ha='center', fontsize=10, color=LABELGREY)
# OR text vertical and single-line so it fits inside the bar
ax.text(3, 10.0, 'OR=3.54 vs Neither', ha='center', va='center', rotation=90,
        fontsize=12, color='white', fontweight='bold')
ax.set_xticks(range(4)); ax.set_xticklabels(prog_names, fontsize=12)
ax.set_ylabel('% from metastatic lesions', fontsize=14)
ax.set_ylim(-4, 26)
ax.tick_params(axis='y', labelsize=12)
panel_title(ax, 'F', 'DUAL-program cells are\nmost metastatic-prone')
despine(ax)
# =====================================================================
# ROW 3: SPECIFICITY
# =====================================================================
# G: AUC comparison
ax = fig.add_subplot(gs[2, 0])
sig_names = ['Bridge\n(27 genes)', 'Leiden 2\n(19 genes)', 'Generic\nproliferation',
             'Generic\nEMT', 'Bottleneck\nnetwork', 'GEM\nresistance',
             'General\nresistance', 'Axis\n(3 genes)']
sig_aucs = [0.630, 0.620, 0.575, 0.572, 0.555, 0.552, 0.516, 0.515]
sig_cols_f = [PURPLE, PURPLE, GREY, GREY, ORANGE, GREY, GREY, GREY]
y_pos = np.arange(len(sig_names))
ax.barh(y_pos, sig_aucs, color=sig_cols_f, height=0.6, edgecolor='white')
ax.axvline(0.5, ls=':', c='grey', lw=0.8)
for i, v in enumerate(sig_aucs):
    ax.text(v+0.004, i, f'{v:.3f}', va='center', fontsize=13,
            fontweight='bold' if v >= 0.62 else 'normal')
ax.set_yticks(y_pos); ax.set_yticklabels(sig_names, fontsize=13)
ax.set_xlabel('AUC (met. vs naïve, library-corrected)', fontsize=14)
ax.set_xlim(0.47, 0.675)
ax.tick_params(axis='x', labelsize=12)
panel_title(ax, 'G', 'Signature specificity\n(metastatic prediction)')
ax.invert_yaxis()
despine(ax)
# H: Gene expression heatmap
ax = fig.add_subplot(gs[2, 1])
genes_heat = ['CDK1','CDKN1A','WEE1','RRM2','TK1','CLSPN',
              'FN1','TGFBI','S100A4','BIRC5']
gem_cyc_m = gem_m & cycling
gem_g1_m = gem_m & (adata_epi.obs['phase']=='G1')
folf_cyc_m = folf_m & cycling
heat_cols_d = {'Normal': normal_m, 'Naïve': naive_primary,
               'GEM G1': gem_g1_m, 'GEM cyc': gem_cyc_m,
               'FOLF cyc': folf_cyc_m, 'Met': met_m}
heat_matrix = np.zeros((len(genes_heat), len(heat_cols_d)))
for j, (tname, tmask) in enumerate(heat_cols_d.items()):
    for i, gene in enumerate(genes_heat):
        heat_matrix[i,j] = get_expr(adata_epi[tmask], gene).mean()
heat_fc = np.zeros_like(heat_matrix)
for i in range(heat_matrix.shape[0]):
    baseline = heat_matrix[i, 0] + 0.01
    for j in range(heat_matrix.shape[1]):
        heat_fc[i, j] = np.log2((heat_matrix[i, j] + 0.01) / baseline)
im = ax.imshow(heat_fc, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
ax.set_xticks(range(len(heat_cols_d)))
ax.set_xticklabels(heat_cols_d.keys(), fontsize=12.5, rotation=30, ha='right')
ax.set_yticks(range(len(genes_heat)))
ax.set_yticklabels(genes_heat, fontsize=13, style='italic')
cb = plt.colorbar(im, ax=ax, shrink=0.65, aspect=20, pad=0.02)
cb.set_label('log₂FC vs normal', fontsize=14)
cb.ax.tick_params(labelsize=12)
ax.axhline(5.5, ls='-', c='white', lw=2)
panel_title(ax, 'H', 'Axis gene expression\nacross conditions')
# I: Axis co-detection rates
ax = fig.add_subplot(gs[2, 2])
det_groups = ['Normal', 'Naïve\ncycling', 'GEM\ncycling', 'FOLFIR\ncycling',
              'Met\ncycling']
det_masks = [normal_m,
             naive_primary & cycling,
             gem_cyc_m, folf_cyc_m,
             met_m & cycling]
det_cols = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]
triple_pcts = []; coexpr_pcts_k = []
for mask in det_masks:
    c1 = get_expr(adata_epi[mask], 'CDK1') > 0
    ca = get_expr(adata_epi[mask], 'CDKN1A') > 0
    w = get_expr(adata_epi[mask], 'WEE1') > 0
    triple_pcts.append((c1 & ca & w).mean() * 100)
    coexpr_pcts_k.append((c1 & ca).mean() * 100)
x = np.arange(len(det_groups))
w_bar = 0.35
ax.bar(x - w_bar/2, coexpr_pcts_k, w_bar, color=det_cols, alpha=0.5,
       label='CDK1⁺/CDKN1A⁺')
ax.bar(x + w_bar/2, triple_pcts, w_bar, color=det_cols,
       label='C⁺A⁺W⁺')
# value labels vertical above each bar → no more twin-label collision
for i in range(len(det_groups)):
    ax.text(i - w_bar/2, coexpr_pcts_k[i]+0.3, f'{coexpr_pcts_k[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')
    ax.text(i + w_bar/2, triple_pcts[i]+0.3, f'{triple_pcts[i]:.1f}%',
            ha='center', va='bottom', rotation=90, fontsize=11.5, fontweight='bold')
ax.set_ylim(0, max(coexpr_pcts_k) * 1.35)     # headroom for vertical labels
ax.set_xticks(x); ax.set_xticklabels(det_groups, fontsize=11.5)
ax.set_ylabel('Co-detection rate (%)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.16),
          ncol=2, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'I', 'Axis co-expression\nacross conditions')
despine(ax)
# =====================================================================
# ROW 4: TREATMENT
# =====================================================================
# J: Drug-specific axis engagement
ax = fig.add_subplot(gs[3, 0])
components = ['CDK1⁺\n(drive)', 'CDKN1A⁺\n(bridge)', 'WEE1⁺\n(brake)', 'DUAL\nprogram']
naive_pct = [8.8, 17.8, 13.8, 13.5]
gem_pct = [3.9, 11.9, 22.8, 3.7]
folf_pct = [16.0, 17.2, 20.7, 28.9]
x = np.arange(len(components))
w = 0.25
ax.bar(x - w, naive_pct, w, color='#85C1E9', label='Naïve cycling')
ax.bar(x, gem_pct, w, color='#9B59B6', label='GEM cycling')
ax.bar(x + w, folf_pct, w, color='#E67E22', label='FOLFIR cycling')
for i, (g, n) in enumerate(zip(gem_pct, naive_pct)):
    if g > n * 1.3:
        ax.text(i, g+1.3, '↑', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')
    elif g < n * 0.7:
        ax.text(i, g+1.3, '↓', ha='center', fontsize=15, color='#9B59B6', fontweight='bold')
for i, (f, n) in enumerate(zip(folf_pct, naive_pct)):
    if f > n * 1.3:
        ax.text(i + w, f+1.3, '↑', ha='center', fontsize=15, color='#E67E22', fontweight='bold')
ax.set_ylim(0, max(folf_pct) * 1.20)
ax.set_xticks(x); ax.set_xticklabels(components, fontsize=12)
ax.set_ylabel('% of cycling cells', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.22),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'J', 'Drug-specific axis engagement\n(GEM: brake ↑ | FOLFIR: drive ↑)')
despine(ax)
# K: Quiescence + WEE1 brake
ax = fig.add_subplot(gs[3, 1])
treat_names = ['Normal', 'Naïve', 'GEM', 'FOLFIR', 'Met.']
cyc_pcts = [45.7, 31.6, 27.0, 37.6, 41.4]
treat_cols_n = ['#AEB6BF','#85C1E9','#9B59B6','#E67E22',RED]
bars = ax.bar(range(len(treat_names)), cyc_pcts, color=treat_cols_n,
              width=0.6, edgecolor='white')
for i, v in enumerate(cyc_pcts):
    ax.text(i, v+1.3, f'{v:.1f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold')
wee1_g1_pcts = []
for name, mask in [('Normal', normal_m), ('Naïve', naive_primary),
                     ('GEM', gem_m), ('FOLFIR', folf_m), ('Met', met_m)]:
    g1_mask = mask & (adata_epi.obs['phase']=='G1')
    if g1_mask.sum() > 10:
        w_det = (get_expr(adata_epi[g1_mask], 'WEE1') > 0).mean() * 100
        wee1_g1_pcts.append(w_det)
    else:
        wee1_g1_pcts.append(0)
ax2 = ax.twinx()
ax2.plot(range(len(treat_names)), wee1_g1_pcts, 's--', color='darkblue',
         ms=9, lw=2, label='WEE1⁺ in G1')
ax2.set_ylim(top=max(wee1_g1_pcts) * 1.32)     # headroom first
# per-marker manual placement (offset points): (dx, dy, ha, va)
#   Normal, Naïve → above; GEM → above; FOLFIR(18%) → below; Met → further below
blue_offsets = [
    (0,  9, 'center', 'bottom'),   # Normal  9%  above
    (0,  -25, 'center', 'bottom'),   # Naïve  15%  above
    (0, 11, 'center', 'bottom'),   # GEM    31%  above the square
    (0,-13, 'center', 'top'),      # FOLFIR 18%  below the square
    (0,-25, 'center', 'top'),      # Met    31%  further below the square
]
for i, v in enumerate(wee1_g1_pcts):
    dx, dy, ha, va = blue_offsets[i]
    ax2.annotate(f'{v:.0f}%', (i, v), textcoords='offset points',
                 xytext=(dx, dy), ha=ha, va=va, fontsize=11.5,
                 color='darkblue', fontweight='semibold')
ax2.set_ylabel('WEE1⁺ in G1 (%)', fontsize=13, color='darkblue')
ax2.tick_params(axis='y', labelsize=11, colors='darkblue')
ax2.spines['top'].set_visible(False)
ax.set_ylim(0, max(cyc_pcts) * 1.20)
ax.set_xticks(range(len(treat_names)))
ax.set_xticklabels(treat_names, fontsize=12)
ax.set_ylabel('% cycling (S+G2M)', fontsize=14)
ax.tick_params(axis='y', labelsize=12)
ax2.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
           framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'K', 'GEM: quiescence + WEE1 brake\n(73% G1, WEE1⁺ in quiescent)')
despine(ax)
# L: Bottleneck enrichment
ax = fig.add_subplot(gs[3, 2])
pcts_plot = [50, 60, 70, 75, 80, 85, 90, 95]
enr_data = {
    'GEM cycling': ([50.2,39.2,30.8,28.2,24.4,20.1,13.8,5.9], '#9B59B6'),
    'FOLFIR cycling': ([54.9,45.8,39.2,35.6,30.6,25.1,18.0,9.7], '#E67E22'),
    'Metastatic': ([55.5,48.2,39.3,31.6,24.9,18.8,13.6,8.7], RED),
}
expected = [50, 40, 30, 25, 20, 15, 10, 5]
for name, (vals, col) in enr_data.items():
    fold = [v/e for v, e in zip(vals, expected)]
    ax.plot(pcts_plot, fold, 'o-', color=col, lw=2.5, ms=7, label=name)
ax.axhline(1, ls=':', c='grey', lw=0.8)
ax.set_xlabel('Axis score percentile threshold', fontsize=14)
ax.set_ylabel('Fold enrichment vs naïve', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.set_ylim(0.7, 2.5)
ax.legend(fontsize=11.5, loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=3, framealpha=0.95, edgecolor='#CCCCCC')
panel_title(ax, 'L', 'Bottleneck enrichment\nin survivors & metastasis')
despine(ax)
ax.annotate('FOLFIR P90:\nOR=1.97', xy=(90, 1.80), xytext=(69, 2.28),
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=1.5),
            fontsize=12, color='#E67E22', fontweight='bold')
ax.annotate('Met P90:\nOR=1.42', xy=(90, 1.36), xytext=(69, 1.60),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5),
            fontsize=12, color=RED, fontweight='bold')
# =====================================================================
# SUPTITLE + ROW LABELS
# =====================================================================
fig.suptitle(
    'External validation: CDK1–CDKN1A–WEE1 bottleneck axis\n'
    '300,577 epithelial cells · 231 patients · 12 independent studies',
    fontsize=19, fontweight='bold', y=1.005, fontfamily='sans-serif')
row_labels = [
    'Metastatic\nassociation',
    'Bridge\nmechanism',
    'Signature\nspecificity',
    'Treatment\npatterns',
]
for i, label in enumerate(row_labels):
    fig.text(1.005, 0.86 - i*0.235, label, fontsize=13, fontweight='bold',
             rotation=270, va='center', ha='center', color=DARKGREY,
             style='italic', fontfamily='sans-serif')
# =====================================================================
# SAVE 1200 DPI
# =====================================================================
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised_1200dpi.png", dpi=1200,
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(f"{OUTPUT_DIR}/Fig5_axis-validation_revised.pdf",
            bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Saved 1200 DPI validation figure to {OUTPUT_DIR}/")